# Scene-aware Video RAG

Index a video, retrieve relevant evidence, and answer questions using a two-column, three-row frame grid. Install `requirements.txt` and start Ollama with the `llava` model before running the cells. Set `VIDEO` to your input file and edit `question` for your video. See the [README](README.md) for setup and EgoSchema evaluation commands.


In [ ]:
from pathlib import Path

from video_processing import Config
from video_rag import VideoRAG

PROJECT = Path.cwd()
VIDEO = PROJECT / "data/videos/example.mp4"
INDEX = PROJECT / "data/indexes/example"

## Index a video

Shots are detected before frames and word-level speech are assigned to chunks. Visual captions make quiet scenes searchable with BM25. To skip audio, set `transcribe=False`.


In [ ]:
config = Config(max_chunk_seconds=30.0, frame_interval=3.0)
rag = VideoRAG.ingest(VIDEO, INDEX, config)
[(chunk.id, chunk.start, chunk.end, chunk.text) for chunk in rag.chunks[:5]]

## Retrieve and answer

The saved grid contains up to six frames in timestamp order, read left to right, top to bottom.


In [ ]:
from IPython.display import display
from PIL import Image

question = "What does the person do after opening the drawer?"
grid_path = PROJECT / "outputs/evidence.jpg"
result = rag.answer(question, mode="hybrid", grid_path=grid_path)
print(result["answer"])
with Image.open(grid_path) as grid:
    display(grid.copy())
result["sources"]

## Compare retrieval strategies

These rankings use the same index. Run the EgoSchema CLI with separate output directories to measure QA accuracy for each mode.


In [ ]:
for mode in ("dense", "bm25", "hybrid"):
    hits = rag.retriever.retrieve(question, mode=mode)
    print(mode, [(hit.chunk.id, hit.ranks) for hit in hits])

## Reload an index


In [ ]:
rag = VideoRAG.load(INDEX)